# Institutional Accumulation Screener

Finds stocks that institutions appear to be **accumulating** — or **distributing** —
using the price-range and volume signature described in *"Institutional Investors
Move Markets. Here's How to Read Their Signs."* (WSJ / IBD Insights, Sept. 7 2026).

---

### How to run this

1. **Runtime → Run all** (top menu), or press **Ctrl+F9** / **⌘+F9**.
2. If Colab warns the notebook wasn't authored by Google, click **Run anyway**.
3. Wait 5–10 minutes. Almost all of that is downloading price history.
4. The finished report appears at the bottom **and downloads to your computer** as
   `institutional_screener.html` — double-click it to open in any browser.

Nothing to install, no terminal, no Python on your machine. To narrow the scan,
change the settings in step 4 and re-run steps 4 and 5.

**Want a fast test first?** In step 4 set `limit` to `50`. That scans 50 names in
about a minute so you can see the whole thing work before committing to the full run.


In [ ]:
#@title Step 1 — install the data library  *(~20 seconds)* { display-mode: "form" }
!pip install -q yfinance
import yfinance, pandas, numpy
print(f"Ready. yfinance {yfinance.__version__}, pandas {pandas.__version__}")


In [ ]:
#@title Step 2 — the report builder  *(double-click to read the code)*
"""Renders the screener payload into one self-contained HTML file."""

from __future__ import annotations

import json

TEMPLATE = r"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Institutional Accumulation Screener</title>
<style>
  :root {
    color-scheme: light;
    --plane:#f9f9f7; --surface:#fcfcfb; --raised:#ffffff;
    --ink:#0b0b0b; --ink-2:#52514e; --ink-3:#898781;
    --rule:#e1e0d9; --ring:rgba(11,11,11,.10);
    --acc:#2a78d6; --acc-soft:#cde2fb; --acc-mid:#86b6ef;
    --good:#0ca30c; --warn:#fab219; --serious:#ec835a; --crit:#d03b3b;
    --good-ink:#006300; --crit-ink:#a32222;
    --wash:rgba(11,11,11,.04);
  }
  @media (prefers-color-scheme: dark) {
    :root:not([data-theme="light"]) {
      color-scheme: dark;
      --plane:#0d0d0d; --surface:#1a1a19; --raised:#222221;
      --ink:#ffffff; --ink-2:#c3c2b7; --ink-3:#898781;
      --rule:#2c2c2a; --ring:rgba(255,255,255,.10);
      --acc:#3987e5; --acc-soft:#184f95; --acc-mid:#256abf;
      --good-ink:#0ca30c; --crit-ink:#e66767;
      --wash:rgba(255,255,255,.05);
    }
  }
  :root[data-theme="dark"] {
    color-scheme: dark;
    --plane:#0d0d0d; --surface:#1a1a19; --raised:#222221;
    --ink:#ffffff; --ink-2:#c3c2b7; --ink-3:#898781;
    --rule:#2c2c2a; --ring:rgba(255,255,255,.10);
    --acc:#3987e5; --acc-soft:#184f95; --acc-mid:#256abf;
    --good-ink:#0ca30c; --crit-ink:#e66767;
    --wash:rgba(255,255,255,.05);
  }

  * { box-sizing: border-box; }
  body {
    margin:0; background:var(--plane); color:var(--ink);
    font:14px/1.5 system-ui,-apple-system,"Segoe UI",sans-serif;
    -webkit-font-smoothing:antialiased;
  }
  .wrap { max-width:1280px; margin:0 auto; padding:28px 20px 80px; }

  header.top { display:flex; justify-content:space-between; align-items:flex-start;
    gap:20px; flex-wrap:wrap; margin-bottom:22px; }
  h1 { font-size:22px; letter-spacing:-.01em; margin:0 0 4px; font-weight:650; }
  .sub { color:var(--ink-2); font-size:13px; margin:0; }
  .sub b { color:var(--ink); font-weight:600; }
  .themebtn { background:var(--surface); border:1px solid var(--rule); color:var(--ink-2);
    border-radius:8px; padding:7px 12px; font:inherit; font-size:12.5px; cursor:pointer; }
  .themebtn:hover { background:var(--wash); }

  /* ---- market health ---- */
  .health { display:grid; grid-template-columns:repeat(auto-fit,minmax(260px,1fr));
    gap:12px; margin-bottom:14px; }
  .tile { background:var(--surface); border:1px solid var(--rule); border-radius:12px;
    padding:14px 16px; display:flex; gap:14px; align-items:center; }
  .dot { width:10px; height:10px; border-radius:50%; flex:0 0 auto; }
  .tile .big { font-size:26px; font-weight:650; line-height:1.1; }
  .tile .lbl { font-size:12px; color:var(--ink-3); text-transform:uppercase;
    letter-spacing:.05em; }
  .tile .note { font-size:12.5px; color:var(--ink-2); }
  .s-good{color:var(--good-ink)} .s-warn{color:var(--serious)} .s-crit{color:var(--crit-ink)}
  .bg-good{background:var(--good)} .bg-warn{background:var(--warn)} .bg-crit{background:var(--crit)}

  .warn { background:var(--surface); border:1px solid var(--rule);
    border-left:3px solid var(--warn); border-radius:12px; padding:13px 16px;
    margin-bottom:14px; font-size:12.5px; color:var(--ink-2); line-height:1.55; }
  .warn b { color:var(--ink); }
  details.about { background:var(--surface); border:1px solid var(--rule);
    border-radius:12px; padding:0 16px; margin-bottom:18px; }
  details.about summary { cursor:pointer; padding:13px 0; font-weight:600; font-size:13.5px;
    list-style:none; display:flex; align-items:center; gap:8px; }
  details.about summary::-webkit-details-marker { display:none; }
  details.about summary::before { content:"›"; display:inline-block; transition:transform .15s;
    color:var(--ink-3); font-size:17px; }
  details.about[open] summary::before { transform:rotate(90deg); }
  .about-body { padding:2px 0 16px; color:var(--ink-2); font-size:13px; max-width:78ch; }
  .about-body dt { font-weight:600; color:var(--ink); margin-top:11px; }
  .about-body dd { margin:2px 0 0; }
  .about-body dl { margin:0; }

  /* ---- controls ---- */
  .controls { display:flex; gap:10px; flex-wrap:wrap; align-items:center;
    margin-bottom:14px; }
  input[type=search], select {
    background:var(--surface); border:1px solid var(--rule); color:var(--ink);
    border-radius:8px; padding:8px 10px; font:inherit; font-size:13px; }
  input[type=search] { min-width:190px; }
  .chips { display:flex; gap:6px; flex-wrap:wrap; }
  .chip { background:var(--surface); border:1px solid var(--rule); color:var(--ink-2);
    border-radius:999px; padding:7px 13px; font-size:12.5px; cursor:pointer;
    white-space:nowrap; }
  .chip[aria-pressed="true"] { background:var(--acc); border-color:var(--acc);
    color:#fff; font-weight:600; }
  .count { margin-left:auto; color:var(--ink-3); font-size:12.5px;
    font-variant-numeric:tabular-nums; }

  /* ---- table ---- */
  .tablecard { background:var(--surface); border:1px solid var(--rule);
    border-radius:12px; overflow-x:auto; }
  table { width:100%; border-collapse:collapse; font-variant-numeric:tabular-nums; }
  thead th { position:sticky; top:0; background:var(--surface); z-index:2;
    text-align:right; font-size:11px; letter-spacing:.05em; text-transform:uppercase;
    color:var(--ink-3); font-weight:600; padding:11px 10px; white-space:nowrap;
    border-bottom:1px solid var(--rule); cursor:pointer; user-select:none; }
  thead th:first-child, thead th.l { text-align:left; }
  thead th:hover { color:var(--ink); }
  thead th .arw { opacity:.45; font-size:9px; }
  tbody td { padding:9px 10px; text-align:right; border-bottom:1px solid var(--rule);
    white-space:nowrap; }
  tbody td.l { text-align:left; }
  tbody tr.row { cursor:pointer; }
  tbody tr.row:hover { background:var(--wash); }
  .tk { font-weight:650; letter-spacing:-.01em; }
  .nm { color:var(--ink-3); font-size:12px; max-width:220px; overflow:hidden;
    text-overflow:ellipsis; display:block; }
  .scorecell { display:flex; align-items:center; gap:9px; justify-content:flex-end; }
  .bar { width:58px; height:7px; background:var(--wash); border-radius:4px;
    overflow:hidden; flex:0 0 auto; }
  .bar > i { display:block; height:100%; background:var(--acc); border-radius:4px; }
  .scoreval { font-weight:650; min-width:34px; text-align:right; }
  .strip { display:inline-flex; gap:2px; }
  .cell { width:15px; height:17px; border-radius:3px; background:var(--wash);
    color:var(--ink-3); font-size:9.5px; font-weight:700; line-height:17px;
    text-align:center; }
  .cell.a { background:var(--good); color:#fff; }
  .cell.d { background:var(--crit); color:#fff; }
  .pill { font-size:11px; padding:3px 8px; border-radius:999px; font-weight:600;
    border:1px solid var(--ring); }
  .pos { color:var(--good-ink); } .neg { color:var(--crit-ink); }

  /* ---- detail ---- */
  tr.detail > td { padding:0; border-bottom:1px solid var(--rule); }
  .panel { padding:18px 20px 22px; background:var(--plane); display:grid;
    grid-template-columns:minmax(300px,1.1fr) minmax(300px,1fr); gap:26px 40px;
    text-align:left; align-items:start; white-space:normal; }
  .panel h4 { margin:0 0 12px; font-size:11px; letter-spacing:.06em;
    text-transform:uppercase; color:var(--ink-3); font-weight:600; }
  .comp { display:grid; grid-template-columns:1fr auto; gap:5px 12px;
    align-items:baseline; font-size:12.5px; }
  .comp .lab { color:var(--ink-2); text-align:left; }
  .comp .track { grid-column:1/-1; height:6px; background:var(--wash);
    border-radius:3px; margin:-2px 0 7px; overflow:hidden; }
  .comp .track > i { display:block; height:100%; background:var(--acc-mid);
    border-radius:3px; }
  .comp .val { font-weight:600; font-variant-numeric:tabular-nums; }
  .wk { display:flex; gap:4px; align-items:flex-end; height:78px;
    padding-bottom:2px; }
  .wk .col { flex:1; display:flex; flex-direction:column; align-items:center;
    gap:4px; justify-content:flex-end; }
  .wk .mark { width:100%; border-radius:3px 3px 0 0; background:var(--acc-mid);
    min-height:3px; }
  .wk .mark.a { background:var(--good); } .wk .mark.d { background:var(--crit); }
  .tick { font-size:10.5px; color:var(--ink-3); line-height:1.45; }
  .wk .tick { font-size:9.5px; }
  .verdict { margin-top:14px; font-size:13px; color:var(--ink-2);
    border-left:2px solid var(--acc); padding-left:11px; }
  .kv { display:grid; grid-template-columns:1fr auto; gap:5px 20px; font-size:12.5px;
    margin:16px 0 0; }
  .kv dt { color:var(--ink-2); }
  .kv dd { margin:0; font-variant-numeric:tabular-nums; font-weight:600;
    text-align:right; }
  .kv dt, .kv dd { padding-bottom:5px; border-bottom:1px solid var(--rule); }
  footer { margin-top:22px; color:var(--ink-3); font-size:12px; max-width:80ch; }
  .empty { padding:44px; text-align:center; color:var(--ink-3); }
  @media (max-width:820px) { .panel { grid-template-columns:1fr; } }
</style>
</head>
<body>
<div class="wrap">

<header class="top">
  <div>
    <h1>Institutional Accumulation Screener</h1>
    <p class="sub">Reading price range and volume for institutional footprints &middot;
      <b id="uni"></b> &middot; generated <b id="gen"></b></p>
  </div>
  <button class="themebtn" id="theme">Toggle theme</button>
</header>

<section class="health" id="health"></section>

<details class="about">
  <summary>How the score is built</summary>
  <div class="about-body">
    <p style="margin-top:0">Institutions are too big to hide. Their buying and selling
    leaves a signature in a stock's <b>trading range</b> and <b>volume</b>. Six components,
    100 points total:</p>
    <dl>
      <dt>Up/down volume ratio, 50 days &mdash; 30 pts</dt>
      <dd>Volume on up days divided by volume on down days. Above 2.0 is strong
      accumulation, 1.0 is churn, below 1.0 points to heavier selling.</dd>
      <dt>Weekly range + volume &mdash; 25 pts</dt>
      <dd>Every one of the last eight weeks contributes where it closed in its own
      range, weighted by how heavy its volume was (the <b>weekly pressure</b> figure,
      running &minus;2 to +2). Weeks clearing the thresholds outright are marked
      <b>A</b> or <b>D</b> in the table, and consecutive A weeks earn a bonus &mdash;
      the pattern repeating is the signal, not any single week.</dd>
      <dt>A/D line slope, 25 days &mdash; 20 pts</dt>
      <dd>The classic Accumulation/Distribution line, its slope normalized by average
      volume so stocks of different size compare.</dd>
      <dt>Pullback volume drying &mdash; 10 pts</dt>
      <dd>When a stock pulls back on <i>lighter</i> volume, institutions are taking a
      break from buying rather than selling. Scored against the field's own spread, so
      the median name lands mid-scale.</dd>
      <dt>Volume expansion &mdash; 5 pts</dt>
      <dd>50-day average volume against the 200-day. No footprints, no institutions.</dd>
      <dt>Trend / support &mdash; 10 pts</dt>
      <dd>How far price sits above its 50- and 200-day averages and below its 52-week
      high &mdash; measured continuously, not as pass/fail. Accumulation only counts if
      the support floor is holding, but a stock 0.1% above its 50-day has not earned what
      one 20% above has.</dd>
    </dl>
    <p><b>Distribution days</b> (the market tiles above, and the DD column) count sessions
    closing lower on higher volume than the day before. The indexes use the article's
    literal 0.2%; individual stocks use the same rule expressed in their own volatility
    (0.22 sigma), because a 0.2% down day is a real event for an index and noise for a
    single stock. A few is noise; many in a short period is an early warning that
    institutions are stepping back.</p>

    <h4 style="margin:18px 0 6px;font-size:13.5px">Two things worth understanding before you sort by score</h4>

    <p><b>Bands are relative to this scan, not absolute.</b> "Strong Accumulation" means
    the top 5% of what was screened; Accumulation the next 15%; the middle 40% Neutral;
    then 25% Distribution and the bottom 15% Heavy Distribution. Fixed score cutoffs
    sound more objective and are in practice worse &mdash; the first version of this tool
    used them and labelled the median stock "Distribution" in a market where two thirds
    of names were above their 200-day average.</p>

    <p><b>Ex-momentum is the column to read second.</b> The composite score correlates
    heavily with recent price performance, partly by construction: a stock that rose
    necessarily had more up days, and up days carry the volume. Ex-momentum is what is
    left after the 63-day return is regressed out of the score &mdash; a positive figure
    means more institutional footprint than this stock's own price action would predict.
    Sorting by score finds what has already worked. Sorting by ex-momentum finds what
    may not have shown up in the price yet, which is the harder and more useful question.</p>

    <p><b>Sect %ile ranks a name inside its own sector.</b> A single ranking of the whole
    market hands you whichever sector the macro currently favours &mdash; four refiners in
    the top fourteen is one bet on crack spreads, not four independent institutional
    decisions. Sorting by Sect %ile, or capping the list with <b>Best N per sector</b>,
    answers the more useful question: is money moving into this name relative to the
    other places it could sit within the same sector? The cap applies after sorting, so
    it works on the ex-momentum ordering too.</p>

    <p><b>The Article column is a second, independent ranking.</b> It scores each name
    using only the tests the article itself names &mdash; up/down volume ratio, consecutive
    heavy-volume closes high in the weekly range, pullbacks on lighter volume, a holding
    support floor, few distribution days &mdash; with none of this tool's own additions.
    It also rewards a stock that has <i>already run</i>, because the article's worked
    example was up 500% on the year and still being accumulated. Where the two columns
    agree, the case is strong on both readings. Where they disagree, the disagreement is
    the interesting part: a high Article score with a low composite usually means an
    A/D line that is not confirming, and the reverse usually means a beaten-down name
    that the article's momentum-continuation framing would never have picked.</p>

    <p>A dot beside a signal means the name was <b>capped by the A/D gate</b>: it scored
    into an accumulation band while its Accumulation/Distribution line was falling, or
    with up/down volume below 1.0. Those two conditions contradict the label, so it is
    downgraded to Neutral rather than presented as a buy candidate.</p>
  </div>
</details>

<div class="controls">
  <input type="search" id="q" placeholder="Ticker or company" aria-label="Search">
  <div class="chips" id="chips"></div>
  <select id="sector" aria-label="Sector"></select>
  <select id="persector" aria-label="Limit per sector">
    <option value="0">All names</option>
    <option value="1">Best 1 per sector</option>
    <option value="2">Best 2 per sector</option>
    <option value="3">Best 3 per sector</option>
    <option value="5">Best 5 per sector</option>
  </select>
  <select id="persub" aria-label="Limit per sub-industry">
    <option value="0">All sub-industries</option>
    <option value="1">Best 1 per sub-industry</option>
    <option value="2">Best 2 per sub-industry</option>
  </select>
  <span class="count" id="count"></span>
</div>

<div class="tablecard">
  <table id="tbl">
    <thead><tr>
      <th class="l" data-k="rank">#<span class="arw"></span></th>
      <th class="l" data-k="ticker">Stock<span class="arw"></span></th>
      <th data-k="score">Score<span class="arw"></span></th>
      <th data-k="classification">Signal<span class="arw"></span></th>
      <th data-k="ex_mom" title="Score minus what the 63-day return alone predicts">Ex-mom<span class="arw"></span></th>
      <th data-k="art_score" title="Scored only on the tests the article itself names">Article<span class="arw"></span></th>
      <th data-k="ud_ratio">U/D vol<span class="arw"></span></th>
      <th class="l" data-k="acc_weeks">8-week pattern<span class="arw"></span></th>
      <th data-k="ad_slope">A/D slope<span class="arw"></span></th>
      <th data-k="dist_days">DD<span class="arw"></span></th>
      <th data-k="price">Price<span class="arw"></span></th>
      <th data-k="pct_off_high">Off high<span class="arw"></span></th>
      <th data-k="sector_pctile" title="Percentile within its own sector">Sect %ile<span class="arw"></span></th>
    </tr></thead>
    <tbody id="tb"></tbody>
  </table>
  <div class="empty" id="empty" hidden>Nothing matches those filters.</div>
</div>

<footer>
  Educational tool, not investment advice. Signals describe what price and volume have
  already done; they do not predict what a stock will do next. Sort order is the composite
  score &mdash; always read the component breakdown before acting on a rank.
</footer>
</div>

<script>
const DATA = __PAYLOAD__;
const $ = s => document.querySelector(s);
const fmtM = v => v>=1e9 ? (v/1e9).toFixed(1)+"B" : v>=1e6 ? (v/1e6).toFixed(0)+"M" : (v/1e3).toFixed(0)+"K";
const pct = v => (v==null||!isFinite(v)) ? "&mdash;" : (v*100).toFixed(1)+"%";
const num = (v,d=2) => (v==null||!isFinite(v)) ? "&mdash;" : v.toFixed(d);
const esc = s => String(s).replace(/[&<>"]/g, c => ({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;"}[c]));

$("#gen").textContent = DATA.generated;
$("#uni").textContent = DATA.count + " of " + DATA.scanned + " scanned";
if (DATA.fit && DATA.fit.r2 != null) {
  const n = document.createElement("p");
  n.className = "sub";
  n.style.marginTop = "4px";
  n.innerHTML = `Score explained by 63-day price momentum alone: <b>R&sup2; ${DATA.fit.r2.toFixed(2)}</b>`
    + (DATA.gated ? ` &middot; <b>${DATA.gated}</b> name${DATA.gated===1?"":"s"} capped by the A/D gate` : "")
    + ` &middot; sort by <b>Ex-mom</b> for the part the price hasn't already told you.`;
  $("#uni").closest("p").after(n);
}

/* sub-industry clusters — the sharpest form of the concentration problem */
const SUBC = (DATA.sectors && DATA.sectors.sub_concentration) || [];
if (SUBC.length) {
  const box = document.createElement("div");
  box.className = "warn";
  box.innerHTML = "<b>Clusters in the top " + DATA.sectors.top_n + ".</b> "
    + SUBC.map(c => `<b>${esc(c.sub_industry)}</b>: ${c.tickers.slice(0,6).map(esc).join(", ")}`
        + `${c.tickers.length>6?" …":""} (${c.got} names, ${c.expected} expected)`).join(" · ")
    + ". Same-sub-industry names move on the same driver — count them as one "
    + "position, not several. <b>Best N per sub-industry</b> collapses them.";
  $("#health").after(box);
}

/* concentration warning — one macro bet wearing several tickers */
const CONC = (DATA.sectors && DATA.sectors.concentration) || [];
if (CONC.length) {
  const box = document.createElement("div");
  box.className = "warn";
  box.innerHTML = "<b>Sector concentration in the top " + DATA.sectors.top_n + ".</b> "
    + CONC.map(c => `<b>${esc(c.sector)}</b> holds ${c.got} places against ${c.expected} expected `
        + `(${c.ratio}&times;) — ${c.tickers.slice(0,6).map(esc).join(", ")}`
        + `${c.tickers.length>6?" …":""}`).join(" · ")
    + ". Names clustered like this usually reflect one macro move rather than several "
    + "independent institutional decisions. Use <b>Best N per sector</b>, or sort by "
    + "<b>Sect %ile</b>, to see the list without that tilt.";
  $("#health").after(box);
}

/* market health tiles */
$("#health").innerHTML = DATA.health.map(h => `
  <div class="tile">
    <span class="dot bg-${h.state==='good'?'good':h.state==='warning'?'warn':'crit'}"></span>
    <div>
      <div class="lbl">${esc(h.label)} &middot; distribution days</div>
      <div class="big s-${h.state==='good'?'good':h.state==='warning'?'warn':'crit'}">${h.count}
        <span style="font-size:13px;font-weight:400;color:var(--ink-3)">of last ${DATA.params.dist_day_window}</span></div>
      <div class="note">${esc(h.note)}</div>
    </div>
  </div>`).join("") || '<div class="tile"><div class="note">Index data unavailable.</div></div>';

/* filters */
const BANDS = ["All","Strong Accumulation","Accumulation","Neutral / Churn","Distribution","Heavy Distribution"];
let band = "All", sector = "All", query = "", sortKey = "score", sortDir = -1,
    perSector = 0, perSub = 0;

$("#chips").innerHTML = BANDS.map(b =>
  `<button class="chip" data-b="${esc(b)}" aria-pressed="${b===band}">${esc(b)}</button>`).join("");
$("#chips").onclick = e => {
  const b = e.target.closest(".chip"); if(!b) return;
  band = b.dataset.b;
  [...$("#chips").children].forEach(c => c.setAttribute("aria-pressed", c.dataset.b===band));
  render();
};
const sectors = ["All", ...new Set(DATA.rows.map(r => r.sector).filter(s => s && s!=="Unknown"))].sort((a,b)=>a==="All"?-1:b==="All"?1:a.localeCompare(b));
$("#sector").innerHTML = sectors.map(s => `<option>${esc(s)}</option>`).join("");
$("#sector").onchange = e => { sector = e.target.value; render(); };
$("#persector").onchange = e => { perSector = +e.target.value; render(); };
$("#persub").onchange = e => { perSub = +e.target.value; render(); };
$("#q").oninput = e => { query = e.target.value.trim().toLowerCase(); render(); };

document.querySelectorAll("thead th").forEach(th => th.onclick = () => {
  const k = th.dataset.k;
  if (k === sortKey) sortDir *= -1;
  else { sortKey = k; sortDir = (k==="rank"||k==="ticker"||k==="pct_off_high") ? 1 : -1; }
  render();
});

function strip(p){
  return '<span class="strip">' + (p||[]).map(w =>
    `<span class="cell ${w.kind==='A'?'a':w.kind==='D'?'d':''}" title="${esc(w.week)} — closed ${(w.pos*100).toFixed(0)}% up its range">${w.kind==='-'?'':w.kind}</span>`).join("") + '</span>';
}

function verdict(r){
  let base;
  if (r.gated) base = "Ranked high on the composite, but its A/D line is falling and the label was capped — the components that scored well outvoted the one that most directly measures accumulation. Treat it as unresolved, not as a buy signal.";
  else if (r.churn_flag) base = "Heavy volume with no price progress — churn. Big money is trading it, but neither side is winning. Wait for the range to break.";
  else if (/Strong Accum/.test(r.classification)) base = "Textbook accumulation: up-volume dominant, repeated heavy-volume closes high in the weekly range, support holding.";
  else if (/Accumulation/.test(r.classification)) base = "Accumulation showing, but not on every component. Check which pieces are carrying the score.";
  else if (/Neutral/.test(r.classification)) base = "Mixed. Institutional footprints are inconclusive here.";
  else if (/Heavy Distribution/.test(r.classification)) base = "Heavy distribution. Down-day volume dominates and the range closes are weak.";
  else base = "Distribution signs: supply is being fed into demand on heavy volume.";

  if (isFinite(r.ex_mom)) {
    if (r.ex_mom >= 8) base += " Its footprint is well ahead of what the price has done — the kind of setup this screen exists to find.";
    else if (r.ex_mom <= -8) base += " Most of the score here is the price move itself; the volume evidence is thinner than the rank suggests.";
  }
  return base;
}

function panel(r){
  const max = {"Up/down volume (50d)":30,"Weekly range + volume":25,"A/D line slope (25d)":20,
               "Pullback volume drying":10,"Volume expansion":5,"Trend / support":10};
  const comps = Object.entries(r.components).map(([k,v]) => `
    <div class="lab">${esc(k)}</div><div class="val">${v.toFixed(1)} <span style="color:var(--ink-3);font-weight:400">/ ${max[k]??''}</span></div>
    <div class="track"><i style="width:${Math.max(0,Math.min(100,100*v/(max[k]||1)))}%"></i></div>`).join("");
  const wk = (r.week_pattern||[]).map(w => `
    <div class="col">
      <div class="mark ${w.kind==='A'?'a':w.kind==='D'?'d':''}" style="height:${Math.max(4,w.pos*62)}px"
           title="closed ${(w.pos*100).toFixed(0)}% up its range"></div>
      <div class="tick">${esc(w.week.split(" ")[1])}</div>
    </div>`).join("");
  return `<div class="panel">
    <div>
      <h4>Score components</h4>
      <div class="comp">${comps}</div>
      <div class="verdict">${esc(verdict(r))}</div>
    </div>
    <div>
      <h4>Where it closed in its weekly range &mdash; last ${(r.week_pattern||[]).length} weeks</h4>
      <div class="wk">${wk}</div>
      <div class="tick" style="margin-top:6px">Bar height = where the week closed in its
        range. <b style="color:var(--good-ink)">Green</b> = accumulation week (high close,
        heavy volume) &middot; <b style="color:var(--crit-ink)">red</b> = distribution week
        &middot; blue = ordinary volume.</div>
      <dl class="kv">
        <dt>Article-only score</dt><dd>${isFinite(r.art_score)?r.art_score.toFixed(1)+(r.art_rank?" (rank "+r.art_rank+")":""):"&mdash;"}</dd>
        <dt>Rank within this scan</dt><dd>${isFinite(r.pctile)?r.pctile.toFixed(0)+"th percentile":"&mdash;"}</dd>
        <dt>Rank within ${esc(r.sub_industry&&r.sub_industry!=="Unknown"?r.sub_industry:"its sub-industry")}</dt><dd>${isFinite(r.sub_pctile)?r.sub_pctile.toFixed(0)+"th percentile":"&mdash;"}</dd>
        <dt>Rank within ${esc(r.sector||"its sector")}</dt><dd>${isFinite(r.sector_pctile)?r.sector_pctile.toFixed(0)+"th percentile":"&mdash;"}${isFinite(r.sector_delta)?` (${r.sector_delta>0?"+":""}${r.sector_delta.toFixed(1)} vs sector median)`:""}</dd>
        <dt>Avg daily dollar volume</dt><dd>$${fmtM(r.avg_dollar_vol)}</dd>
        <dt>Ex-momentum (score vs price action)</dt><dd class="${r.ex_mom>0?'pos':r.ex_mom<0?'neg':''}">${isFinite(r.ex_mom)?(r.ex_mom>0?"+":"")+r.ex_mom.toFixed(1)+" pts":"&mdash;"}</dd>
        <dt>Weekly pressure (&minus;2 to +2)</dt><dd>${num(r.week_pressure)}</dd>
        <dt>Up/down volume (50d)</dt><dd>${num(r.ud_ratio)}</dd>
        <dt>Accumulation weeks</dt><dd>${r.acc_weeks} (longest run ${r.acc_streak})</dd>
        <dt>Distribution weeks</dt><dd>${r.dist_weeks}</dd>
        <dt>Pullback volume vs 50d avg</dt><dd>${num(r.pullback_dryness)}&times;</dd>
        <dt>Volume expansion (50d/200d)</dt><dd>${num(r.vol_expansion)}&times;</dd>
        <dt>vs 50-day avg</dt><dd>${pct(r.pct_vs_50dma)}</dd>
        <dt>vs 200-day avg</dt><dd>${pct(r.pct_vs_200dma)}</dd>
        <dt>63-day return</dt><dd>${pct(r.ret_63d)}</dd>
      </dl>
    </div>
  </div>`;
}

function render(){
  let rows = DATA.rows.filter(r =>
    (band==="All" || r.classification===band) &&
    (sector==="All" || r.sector===sector) &&
    (!query || r.ticker.toLowerCase().includes(query) || (r.name||"").toLowerCase().includes(query))
  ).sort((a,b) => {
    const x=a[sortKey], y=b[sortKey];
    if (typeof x === "string") return sortDir * x.localeCompare(y);
    const xv = (x==null||!isFinite(x)) ? -1e18 : x, yv = (y==null||!isFinite(y)) ? -1e18 : y;
    return sortDir * (xv - yv);
  });

  // Cap per sector AFTER sorting, so "best 2 per sector" means best by whatever
  // column is currently sorted — score, ex-momentum, or anything else.
  if (perSector > 0) {
    const seen = {};
    rows = rows.filter(r => {
      const s = r.sector || "Unknown";
      seen[s] = (seen[s] || 0) + 1;
      return seen[s] <= perSector;
    });
  }
  if (perSub > 0) {
    const seen = {};
    rows = rows.filter(r => {
      const s = r.sub_industry || "Unknown";
      if (s === "Unknown") return true;
      seen[s] = (seen[s] || 0) + 1;
      return seen[s] <= perSub;
    });
  }

  $("#count").textContent = rows.length + " stocks"
    + (perSector ? ` · best ${perSector} per sector` : "");
  $("#empty").hidden = rows.length > 0;
  document.querySelectorAll("thead th").forEach(th => {
    th.querySelector(".arw").textContent = th.dataset.k===sortKey ? (sortDir>0?" ▲":" ▼") : "";
  });

  $("#tb").innerHTML = rows.map(r => `
    <tr class="row" data-t="${esc(r.ticker)}">
      <td class="l" style="color:var(--ink-3)">${r.rank}</td>
      <td class="l"><span class="tk">${esc(r.ticker)}</span><span class="nm">${esc(r.name||"")}</span></td>
      <td><div class="scorecell"><div class="bar"><i style="width:${Math.max(2,r.score)}%"></i></div>
          <span class="scoreval">${r.score.toFixed(0)}</span></div></td>
      <td><span class="pill ${/Accum/.test(r.classification)?'pos':/Distrib/.test(r.classification)?'neg':''}"
          title="${r.gated?'Capped by the A/D gate — labelled accumulation while its A/D line was falling':''}">${r.churn_flag?'Churn':esc(r.classification.replace(' / Churn',''))}${r.gated?' •':''}</span></td>
      <td class="${r.ex_mom>0?'pos':r.ex_mom<0?'neg':''}">${r.ex_mom==null||!isFinite(r.ex_mom)?"&mdash;":(r.ex_mom>0?"+":"")+r.ex_mom.toFixed(1)}</td>
      <td title="${r.art_rank?'article rank '+r.art_rank:''}">${isFinite(r.art_score)?r.art_score.toFixed(0):"&mdash;"}</td>
      <td>${num(r.ud_ratio)}</td>
      <td class="l">${strip(r.week_pattern)}</td>
      <td>${num(r.ad_slope)}</td>
      <td>${r.dist_days}</td>
      <td>$${num(r.price)}</td>
      <td>${pct(r.pct_off_high)}</td>
      <td>${isFinite(r.sector_pctile)?r.sector_pctile.toFixed(0):"&mdash;"}</td>
    </tr>`).join("");
}

$("#tb").onclick = e => {
  const tr = e.target.closest("tr.row"); if(!tr) return;
  const nxt = tr.nextElementSibling;
  if (nxt && nxt.classList.contains("detail")) { nxt.remove(); return; }
  document.querySelectorAll("tr.detail").forEach(d => d.remove());
  const r = DATA.rows.find(x => x.ticker === tr.dataset.t);
  const d = document.createElement("tr");
  d.className = "detail";
  d.innerHTML = `<td colspan="13">${panel(r)}</td>`;
  tr.after(d);
};

$("#theme").onclick = () => {
  const cur = document.documentElement.getAttribute("data-theme");
  const dark = cur ? cur === "dark"
    : matchMedia("(prefers-color-scheme: dark)").matches;
  document.documentElement.setAttribute("data-theme", dark ? "light" : "dark");
};

render();
</script>
</body>
</html>
"""


def write_report(payload: dict, path: str) -> str:
    html = TEMPLATE.replace("__PAYLOAD__", json.dumps(payload, default=float))
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(html)
    return path


In [ ]:
#@title Step 3 — the screening engine  *(double-click to read the code)*
#!/usr/bin/env python3
"""
Institutional Accumulation / Distribution Screener
==================================================

Implements the strategy described in "Institutional Investors Move Markets.
Here's How to Read Their Signs." (WSJ / IBD Insights, Sept. 7 2026).

The article's thesis: institutions are too big to hide. Their buying and selling
shows up as a signature in price *range* and *volume*:

  ACCUMULATION  - price closes in the upper part of its weekly range on
                  above-average volume, repeatedly, and holds a support floor.
  DISTRIBUTION  - price closes in the lower part of its range on above-average
                  volume; supply is being fed into strong demand.
  CHURN         - heavy volume, no price progress. Neither side winning.

This script turns that into six measurable components, scores every stock in a
universe 0-100, and writes a self-contained interactive HTML report.

Usage
-----
    pip install yfinance pandas numpy lxml
    python institutional_screener.py                    # S&P 500 + 400 + NDX
    python institutional_screener.py --universe sp500
    python institutional_screener.py --tickers-file my_watchlist.txt
    python institutional_screener.py --min-price 10 --min-dollar-vol 10e6

Outputs (into --outdir, default ./output):
    institutional_screener.html   <- open this
    screen_results.csv
    screen_results.json
"""

from __future__ import annotations

import argparse
import io
import json
import math
import os
import sys
import time
import warnings
from dataclasses import dataclass, asdict, field
from datetime import datetime, timezone

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

try:
    from report import write_report
except ImportError:
    # In the Colab notebook report.py's contents are defined in an earlier cell,
    # so write_report is already in the namespace and there is no module to import.
    pass

# ----------------------------------------------------------------------------
# Tunable parameters. Every threshold the strategy depends on lives here.
# ----------------------------------------------------------------------------

UD_WINDOW = 50          # article: "up/down volume ratio ... over a 50-day period"
WEEK_LOOKBACK = 8       # weeks of range/volume pattern to read
WEEK_VOL_MULT = 1.05    # a week counts as "heavy volume" above this x its 10wk avg
ACC_RANGE_POS = 0.60    # close in the top 40% of the weekly range = accumulation
DIST_RANGE_POS = 0.40   # close in the bottom 40% of the weekly range = distribution
AD_SLOPE_WINDOW = 25    # days for the Accumulation/Distribution line regression
PULLBACK_WINDOW = 15    # days over which to test whether selling volume is drying up
DIST_DAY_WINDOW = 25    # article: "many distribution days in a short period"
DIST_DAY_DROP = -0.002  # article: "closes 0.2% lower or more on higher volume"

# The article's -0.2% distribution-day rule is calibrated for an INDEX, whose
# daily standard deviation runs near 0.9%. That is about 0.22 sigma. Applied
# unchanged to a single stock - three to five times as volatile - a 0.2% down
# day is noise, and nearly every name looks like it is under distribution. So
# single stocks use the same rule expressed in their own sigma; indexes keep the
# literal threshold.
DIST_DAY_SIGMA = 0.22

# Score component maximums (sum = 100)
W_UD_RATIO = 30
W_WEEKLY = 25
W_AD_LINE = 20
W_PULLBACK = 10
W_VOL_EXPANSION = 5
W_TREND = 10

# Component bounds, calibrated against the 10th and 90th percentiles of a live
# 899-name scan so that each component spends its points across the field rather
# than handing most of them to everybody. Before this, pullback-dryness paid out
# 79% of its maximum with a third of names maxed and trend paid 68% with 40%
# maxed - 20 of the 100 points behaving as a constant. A component that almost
# everyone maxes is not scoring, it is padding.
PULLBACK_BOUNDS = (1.10, 0.68)      # (zero-point, full-marks) - lower is drier
VS_50DMA_BOUNDS = (-0.08, 0.10)
VS_200DMA_BOUNDS = (-0.12, 0.23)
OFF_HIGH_BOUNDS = (-0.36, -0.03)

# Classification is by rank within the scan, not by absolute score. Absolute
# cutoffs guessed ahead of time do not survive contact with a real cross-section:
# in the first live run the median stock scored 36 and landed in a band labelled
# "Distribution" while two thirds of the market sat above its 200-day average.
# (cumulative percentile from the bottom, label)
PCTILE_BANDS = [
    (0.95, "Strong Accumulation"),
    (0.80, "Accumulation"),
    (0.40, "Neutral / Churn"),
    (0.15, "Distribution"),
    (0.00, "Heavy Distribution"),
]

# Below this many names a percentile is meaningless, so fall back to absolutes.
PCTILE_MIN_N = 20
CLASS_BANDS = [
    (62, "Strong Accumulation"),
    (50, "Accumulation"),
    (35, "Neutral / Churn"),
    (22, "Distribution"),
    (0,  "Heavy Distribution"),
]

# A name cannot be called accumulation while the measure of accumulation itself
# is falling. In the first live run seven of the top twenty had a flat or
# negative A/D slope; the other components had simply outvoted the one that most
# directly answers the question being asked.
GATE_MIN_AD_SLOPE = 0.0
GATE_MIN_UD_RATIO = 1.0

# Fallback universe if Wikipedia is unreachable. Deliberately short - the most
# liquid US large caps - so the script degrades instead of dying.
FALLBACK_TICKERS = """
AAPL MSFT NVDA AMZN GOOGL GOOG META AVGO TSLA BRK-B LLY JPM V UNH XOM MA COST
HD PG JNJ WMT NFLX CRM BAC ABBV ORCL CVX KO AMD PEP MRK TMO ADBE LIN ACN CSCO
MCD ABT WFC DHR PM TXN GE INTU VZ IBM DIS QCOM AMGN CAT NOW NEE PFE UBER SPGI
CMCSA AMAT RTX HON UNP GS LOW ISRG BKNG COP AXP ELV SYK BLK PGR VRTX MU LRCX
TJX ADI SCHW MDT C REGN BSX PLD ETN MMC ADP CB DE PANW KLAC SO FI CI SBUX BMY
MDLZ ANET UPS ICE ZTS SHW GILD MO CME DUK EQIX WM PYPL CDNS SNPS APH MCK ITW
CSX AON PH TDG MSI NOC CL EOG PNC MMM USB FDX ORLY MAR MPC ROP EMR APD NSC
TGT AJG SLB HLT AFL NXPI PSX TRV DHI ABNB WMB DELL CRWD FTNT ADSK IDXX ROST
"""


# ----------------------------------------------------------------------------
# Universe construction
# ----------------------------------------------------------------------------

WIKI_HEADERS = {"User-Agent": "Mozilla/5.0 (screener; educational use)"}

# Column names vary a little between these tables, so each field lists the
# candidates in order of preference. GICS Sub-Industry is the important one: it
# is what separates four oil refiners from four independent Energy ideas, and it
# has been sitting in these tables all along.
WIKI_SOURCES = {
    "sp500": "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
    "sp400": "https://en.wikipedia.org/wiki/List_of_S%26P_400_companies",
    "sp600": "https://en.wikipedia.org/wiki/List_of_S%26P_600_companies",
    "ndx":   "https://en.wikipedia.org/wiki/Nasdaq-100",
}

WIKI_COLS = {
    "ticker": ["Symbol", "Ticker", "Ticker symbol"],
    "name": ["Security", "Company", "Company name"],
    "sector": ["GICS Sector", "Sector"],
    "sub_industry": ["GICS Sub-Industry", "GICS Sub Industry", "Sub-Industry"],
}

UNIVERSE_SETS = {
    "sp500": ["sp500"],
    "sp400": ["sp400"],
    "sp600": ["sp600"],
    "ndx": ["ndx"],
    "broad": ["sp500", "sp400", "ndx"],           # ~1,000 large and mid caps
    "wide": ["sp500", "sp400", "sp600", "ndx"],   # ~1,600, adds small caps
}


def _pick(table_cols, candidates):
    for c in candidates:
        if c in table_cols:
            return c
    return None


def _read_wiki_table(url: str) -> pd.DataFrame:
    import requests
    resp = requests.get(url, headers=WIKI_HEADERS, timeout=30)
    resp.raise_for_status()
    tables = pd.read_html(io.StringIO(resp.text))
    for t in tables:
        cols = {str(c) for c in t.columns}
        tick = _pick(cols, WIKI_COLS["ticker"])
        name = _pick(cols, WIKI_COLS["name"])
        if not (tick and name):
            continue
        sector = _pick(cols, WIKI_COLS["sector"])
        sub = _pick(cols, WIKI_COLS["sub_industry"])
        if sector is None:          # a table with a ticker but no GICS data is
            continue                # some other table on the page
        return pd.DataFrame({
            "ticker": t[tick].astype(str).str.strip(),
            "name": t[name].astype(str).str.strip(),
            "sector": t[sector].astype(str).str.strip(),
            "sub_industry": (t[sub].astype(str).str.strip()
                             if sub else "Unknown"),
        })
    raise ValueError(f"no matching table at {url}")


def build_universe(which: str) -> pd.DataFrame:
    frames = []
    for key in UNIVERSE_SETS[which]:
        try:
            df = _read_wiki_table(WIKI_SOURCES[key])
            subs = df.sub_industry.nunique()
            print(f"  {key}: {len(df)} names, {subs} sub-industries")
            frames.append(df)
        except Exception as exc:                                  # noqa: BLE001
            print(f"  {key}: FAILED ({exc})")

    if not frames:
        print("  falling back to the bundled large-cap list "
              "(index membership may be stale)")
        tick = FALLBACK_TICKERS.split()
        return pd.DataFrame({"ticker": tick, "name": tick, "sector": "Unknown",
                             "sub_industry": "Unknown"})

    uni = pd.concat(frames, ignore_index=True)
    uni["ticker"] = uni["ticker"].str.replace(".", "-", regex=False).str.upper()
    uni = uni.drop_duplicates(subset="ticker").reset_index(drop=True)
    return uni


def load_tickers_file(path: str) -> pd.DataFrame:
    with open(path) as fh:
        raw = [ln.split("#")[0].strip() for ln in fh]
    tick = [t.upper().replace(".", "-") for t in raw if t]
    return pd.DataFrame({"ticker": tick, "name": tick, "sector": "Unknown",
                         "sub_industry": "Unknown"})


# ----------------------------------------------------------------------------
# Data download
# ----------------------------------------------------------------------------

def download_prices(tickers: list[str], period: str = "2y",
                    batch: int = 100, pause: float = 1.0) -> dict[str, pd.DataFrame]:
    import yfinance as yf
    out: dict[str, pd.DataFrame] = {}
    total = len(tickers)
    for i in range(0, total, batch):
        chunk = tickers[i:i + batch]
        print(f"  downloading {i + 1}-{min(i + batch, total)} of {total} ...",
              flush=True)
        for attempt in range(3):
            try:
                data = yf.download(chunk, period=period, interval="1d",
                                   group_by="ticker", auto_adjust=True,
                                   threads=True, progress=False)
                break
            except Exception as exc:                              # noqa: BLE001
                if attempt == 2:
                    print(f"    batch failed: {exc}")
                    data = None
                else:
                    time.sleep(3 * (attempt + 1))
        if data is None:
            continue
        for t in chunk:
            try:
                df = data[t] if isinstance(data.columns, pd.MultiIndex) else data
            except KeyError:
                continue
            df = df.dropna(subset=["Close", "Volume"])
            if len(df) >= 220:
                out[t] = df
        if i + batch < total:
            time.sleep(pause)
    return out


# ----------------------------------------------------------------------------
# The metrics. Each is a direct translation of one idea in the article.
# ----------------------------------------------------------------------------

@dataclass
class Metrics:
    ticker: str
    name: str = ""
    sector: str = "Unknown"
    sub_industry: str = "Unknown"
    price: float = float("nan")
    avg_dollar_vol: float = float("nan")

    ud_ratio: float = float("nan")        # up/down volume ratio, 50d
    acc_weeks: int = 0                    # heavy-volume weeks closing high in range
    dist_weeks: int = 0                   # heavy-volume weeks closing low in range
    acc_streak: int = 0                   # longest consecutive accumulation run
    week_pressure: float = float("nan")   # continuous weekly range/volume pressure
    ad_slope: float = float("nan")        # A/D line slope, normalized
    pullback_dryness: float = float("nan")  # down-day volume vs 50d average
    vol_expansion: float = float("nan")   # 50d avg volume / 200d avg volume
    dist_days: int = 0                    # volatility-scaled, last 25 sessions

    pct_vs_50dma: float = float("nan")
    pct_vs_200dma: float = float("nan")
    pct_off_high: float = float("nan")
    ret_63d: float = float("nan")

    score: float = 0.0
    art_score: float = 0.0               # the article's own tests, scored alone
    art_rank: int = 0
    ex_mom: float = float("nan")          # score minus what 63d return alone predicts
    pctile: float = float("nan")          # rank within this scan, 0-100
    sector_pctile: float = float("nan")   # rank within its own sector, 0-100
    sector_delta: float = float("nan")    # score minus its sector's median score
    sub_pctile: float = float("nan")      # rank within its own sub-industry
    components: dict = field(default_factory=dict)
    classification: str = ""
    churn_flag: bool = False
    gated: bool = False                   # capped by the A/D gate
    week_pattern: list = field(default_factory=list)  # last 8 weeks: A / D / -


def up_down_volume_ratio(df: pd.DataFrame, window: int = UD_WINDOW) -> float:
    """The article's headline gauge: up-day volume divided by down-day volume."""
    d = df.tail(window + 1)
    chg = d["Close"].diff()
    vol = d["Volume"]
    up = vol[chg > 0].sum()
    down = vol[chg < 0].sum()
    if down <= 0:
        return 9.99 if up > 0 else float("nan")
    return float(min(up / down, 9.99))


def weekly_pattern(df: pd.DataFrame, lookback: int = WEEK_LOOKBACK):
    """Where in its weekly range did the stock close, and on what volume.

    The article's core visual: 'shares closed near the top of their trading
    range during the week ... on heavy volume. This pattern was repeated for
    several consecutive weeks.'
    """
    wk = df.resample("W-FRI").agg({"Open": "first", "High": "max", "Low": "min",
                                   "Close": "last", "Volume": "sum"}).dropna()
    if len(wk) < 14:
        return 0, 0, 0, [], float("nan")

    rng = (wk["High"] - wk["Low"]).replace(0, np.nan)
    pos = ((wk["Close"] - wk["Low"]) / rng).fillna(0.5)
    avg_vol = wk["Volume"].rolling(10).mean().shift(1)
    vol_mult = (wk["Volume"] / avg_vol).clip(0.5, 2.0)
    heavy = wk["Volume"] >= WEEK_VOL_MULT * avg_vol

    acc = (pos >= ACC_RANGE_POS) & heavy
    dist = (pos <= DIST_RANGE_POS) & heavy

    # Continuous weekly pressure. Counting weeks that cross a threshold throws
    # away most of the signal: a week closing at 59% of its range on 1.04x volume
    # scores identically to one closing at 5% on half volume. Every week now
    # contributes its own range position, weighted by how heavy its volume was.
    # Range: about -2 (every week closing at its low on double volume) to +2.
    pressure = float(((pos - 0.5) * 2 * vol_mult).tail(lookback).mean())

    a = acc.tail(lookback)
    d = dist.tail(lookback)
    p = pos.tail(lookback)

    pattern = []
    for ts, is_a, is_d, pv in zip(a.index, a.values, d.values, p.values):
        pattern.append({
            "week": ts.strftime("%b %d"),
            "kind": "A" if is_a else ("D" if is_d else "-"),
            "pos": round(float(pv), 3),
        })

    streak = best = 0
    for is_a in a.values:
        streak = streak + 1 if is_a else 0
        best = max(best, streak)

    return int(a.sum()), int(d.sum()), int(best), pattern, pressure


def ad_line_slope(df: pd.DataFrame, window: int = AD_SLOPE_WINDOW) -> float:
    """Slope of the classic Accumulation/Distribution line, in units of
    'average daily volumes accumulated per day' so it compares across stocks."""
    d = df.tail(window + 60)
    rng = (d["High"] - d["Low"]).replace(0, np.nan)
    mfm = (((d["Close"] - d["Low"]) - (d["High"] - d["Close"])) / rng).fillna(0)
    ad = (mfm * d["Volume"]).cumsum()
    y = ad.tail(window).to_numpy(dtype=float)
    if len(y) < window:
        return float("nan")
    x = np.arange(len(y), dtype=float)
    slope = np.polyfit(x, y, 1)[0]
    denom = float(d["Volume"].tail(50).mean())
    return float(slope / denom) if denom > 0 else float("nan")


def pullback_dryness(df: pd.DataFrame, window: int = PULLBACK_WINDOW) -> float:
    """'The stock pulled back ... but that was accompanied by lower volume,
    indicating that institutions were taking a break from buying.'

    Down-day volume over the recent window, relative to the 50-day average.
    Below 1.0 means selling is happening on thin volume - a healthy pullback.
    """
    d = df.tail(window + 1)
    chg = d["Close"].diff()
    down_vol = d["Volume"][chg < 0]
    base = float(df["Volume"].tail(50).mean())
    if len(down_vol) == 0 or base <= 0:
        return 0.7          # no down days at all reads as maximally dry
    return float(down_vol.mean() / base)


def distribution_days(df: pd.DataFrame, window: int = DIST_DAY_WINDOW,
                      threshold: float | None = None) -> int:
    """'A distribution day occurs when [it] closes 0.2% lower or more on
    higher volume.'

    threshold=None scales that rule to the instrument's own volatility, which is
    what single stocks need. Pass DIST_DAY_DROP for the article's literal index
    figure.
    """
    d = df.tail(window + 1)
    ret = d["Close"].pct_change()
    if threshold is None:
        sigma = float(df["Close"].pct_change().tail(100).std())
        threshold = (-DIST_DAY_SIGMA * sigma
                     if np.isfinite(sigma) and sigma > 0 else DIST_DAY_DROP)
    vol_up = d["Volume"].diff() > 0
    return int(((ret <= threshold) & vol_up).sum())


# ----------------------------------------------------------------------------
# Scoring
# ----------------------------------------------------------------------------

def _lin(x, lo, hi, out_max):
    """Linear map of x from [lo, hi] onto [0, out_max], clipped."""
    if not np.isfinite(x):
        return 0.0
    return float(np.clip((x - lo) / (hi - lo), 0, 1) * out_max)


def score_stock(m: Metrics) -> Metrics:
    c = {}

    # 1. Up/down volume ratio. Scored on a log scale: 0.7 -> 0, 2.5 -> full.
    #    The article calls 2.0 strong, 1.0 churn, below 1.0 distribution.
    r = m.ud_ratio
    c["Up/down volume (50d)"] = (
        _lin(math.log(r), math.log(0.70), math.log(2.50), W_UD_RATIO)
        if np.isfinite(r) and r > 0 else 0.0)

    # 2. Weekly close-in-range pressure (continuous), plus a bonus for
    #    consecutive accumulation weeks ("repeated for several consecutive weeks").
    c["Weekly range + volume"] = (_lin(m.week_pressure, -0.45, 0.55, W_WEEKLY * 0.8)
                                  + _lin(m.acc_streak, 0, 3, W_WEEKLY * 0.2))

    # 3. A/D line direction.
    c["A/D line slope (25d)"] = _lin(m.ad_slope, -0.25, 0.45, W_AD_LINE)

    # 4. Are pullbacks happening on drying volume? Bounds sit at the field's
    #    p90/p10 so the median name lands mid-scale instead of maxing out.
    c["Pullback volume drying"] = _lin(m.pullback_dryness, *PULLBACK_BOUNDS,
                                       W_PULLBACK)

    # 5. Is volume expanding at all? No footprints, no institutions. The band is
    #    centred below 1.0 because a 50-day window sitting inside the summer lull
    #    reads low against a 200-day for purely seasonal reasons - at 0.95 the
    #    old floor half the market scored zero here every August.
    c["Volume expansion"] = _lin(m.vol_expansion, 0.80, 1.30, W_VOL_EXPANSION)

    # 6. Trend/support context. Accumulation only counts if price holds a floor.
    #    Continuous rather than pass/fail: a stock 0.1% above its 50-day average
    #    was previously scored identically to one 20% above, which is why 40% of
    #    the field maxed this component.
    trend = (_lin(m.pct_vs_50dma, *VS_50DMA_BOUNDS, 4)
             + _lin(m.pct_vs_200dma, *VS_200DMA_BOUNDS, 3)
             + _lin(m.pct_off_high, *OFF_HIGH_BOUNDS, 3))
    c["Trend / support"] = trend

    m.components = {k: round(v, 2) for k, v in c.items()}
    m.score = round(float(sum(c.values())), 1)

    # Churn: heavy volume, no progress, ratio pinned near 1.0.
    m.churn_flag = bool(
        np.isfinite(m.vol_expansion) and m.vol_expansion > 1.15
        and np.isfinite(m.ret_63d) and abs(m.ret_63d) < 0.04
        and np.isfinite(m.ud_ratio) and 0.85 <= m.ud_ratio <= 1.20
    )
    return m


def article_score(m: Metrics) -> float:
    """A second, independent ranking built ONLY from what the article asserts.

    The composite score is my construction and reflects choices the article never
    makes - an A/D-line component, a momentum residual, sector neutrality. This
    function deliberately makes none of them. It scores each name against the
    article's own tests, in the article's own emphasis, so the two rankings can
    be compared and their disagreements read.

    The tests, and where they come from:
      "a stock's up/down volume ratio ... A resulting ratio of 2.0 means twice as
       much up volume ... a ratio below 1.0 points to heavier volume on down
       days"                                                        -> 30 points
      "shares closed near the top of their trading range ... on heavy volume.
       This pattern was repeated for several consecutive weeks"     -> 25 points
      "the stock pulled back ... accompanied by lower volume, indicating that
       institutions were taking a break from buying"                -> 15 points
      "institutions can keep a stock's price high ... creating what's called
       support", and its example had already made "an impressive run" -> 20 points
      "many distribution days in a short period is an early warning"  -> 10 points

    Note what this rewards that the composite does not: a stock that has ALREADY
    run. The article's worked example was up 500% on the year and still being
    accumulated. This is a momentum-continuation framework, not a bottom-fishing
    one, and scoring it faithfully means saying so.
    """
    s = 0.0
    # 1. The up/down volume ratio, the article's headline gauge. 1.0 earns
    #    nothing (its definition of churn); 2.0 earns full marks; above that a
    #    little extra, capped.
    if np.isfinite(m.ud_ratio):
        s += float(np.clip((m.ud_ratio - 1.0) / 1.0, 0, 1.4)) * 30

    # 2. Closing high in the weekly range on heavy volume, REPEATED. The
    #    consecutive run carries most of the weight because the article's own
    #    example turns on repetition, not on any single week.
    s += float(np.clip(m.acc_streak / 3, 0, 1)) * 18
    s += float(np.clip((m.acc_weeks - m.dist_weeks) / 4, 0, 1)) * 7

    # 3. Pullbacks arriving on lighter volume.
    if np.isfinite(m.pullback_dryness):
        s += float(np.clip((1.05 - m.pullback_dryness) / 0.30, 0, 1)) * 15

    # 4. The support floor holding, and a run already under way.
    if np.isfinite(m.pct_vs_50dma):
        s += float(np.clip(m.pct_vs_50dma / 0.10, 0, 1)) * 8
    if np.isfinite(m.pct_off_high):
        s += float(np.clip((m.pct_off_high + 0.20) / 0.20, 0, 1)) * 6
    if np.isfinite(m.ret_63d):
        s += float(np.clip(m.ret_63d / 0.30, 0, 1)) * 6

    # 5. Distribution days, few.
    s += float(np.clip((6 - m.dist_days) / 5, 0, 1)) * 10

    # The up/down term is allowed to overshoot its 30 points (the article treats
    # 2.0 as strong, not as a ceiling), so the raw total can pass 100. Clamp it,
    # so the column stays on the same 0-100 scale as the composite and a name
    # can reach 100 on exceptional volume without having to max every other test.
    return round(min(s, 100.0), 1)


def classify(results: list[Metrics]) -> None:
    """Assigns each name a percentile and a band, then applies the A/D gate.

    Bands are relative to the scan, so "Strong Accumulation" always means the top
    5% of what was actually scanned - never an absolute score that may or may not
    be reachable in a given tape. The gate is the absolute check that keeps a
    relative label honest.
    """
    n = len(results)
    if n == 0:
        return

    if n < PCTILE_MIN_N:
        for m in results:
            m.pctile = float("nan")
            for cutoff, label in CLASS_BANDS:
                if m.score >= cutoff:
                    m.classification = label
                    break
    else:
        ranked = sorted(results, key=lambda x: x.score)
        for i, m in enumerate(ranked):
            p = i / (n - 1)
            m.pctile = round(100 * p, 1)
            for cutoff, label in PCTILE_BANDS:
                if p >= cutoff:
                    m.classification = label
                    break

    for m in results:
        if m.classification in ("Strong Accumulation", "Accumulation"):
            ad_ok = np.isfinite(m.ad_slope) and m.ad_slope > GATE_MIN_AD_SLOPE
            ud_ok = np.isfinite(m.ud_ratio) and m.ud_ratio > GATE_MIN_UD_RATIO
            if not (ad_ok and ud_ok):
                m.classification = "Neutral / Churn"
                m.gated = True


def add_momentum_residual(results: list[Metrics]) -> dict:
    """The score correlates ~0.7 with the trailing 63-day return, partly by
    construction: a stock that rose had more up days, and up days carry the
    volume. So report what is left after the return is regressed out - the part
    of the accumulation signal the price has not already told you.

    A positive ex-momentum figure means more institutional footprint than this
    stock's own price action would lead you to expect.

    The regression runs against the *rank* of the 63-day return rather than the
    return itself. Raw returns have a long right tail - one name up 207% drags a
    least-squares line badly and hands quiet stocks residuals of -70 on a scale
    that only spans 100. Rank is bounded, so no single moonshot distorts the rest.
    """
    raw = np.array([m.ret_63d for m in results], dtype=float)
    y = np.array([m.score for m in results], dtype=float)
    ok = np.isfinite(raw) & np.isfinite(y)
    if ok.sum() < PCTILE_MIN_N:
        for m in results:
            m.ex_mom = float("nan")
        return {}

    x = np.full(len(raw), np.nan)
    x[ok] = pd.Series(raw[ok]).rank(pct=True).to_numpy()
    slope, intercept = np.polyfit(x[ok], y[ok], 1)
    r = float(np.corrcoef(x[ok], y[ok])[0, 1])
    for m, xi in zip(results, x):
        m.ex_mom = (round(float(m.score - (slope * xi + intercept)), 1)
                    if np.isfinite(xi) else float("nan"))
    return {"slope": float(slope), "intercept": float(intercept),
            "r2": round(r * r, 3), "n": int(ok.sum())}


SECTOR_MIN_N = 8        # below this a within-sector percentile is meaningless
SUB_MIN_N = 5           # sub-industries are smaller, so a lower floor


def add_sector_ranks(results: list[Metrics], top_n: int = 50) -> dict:
    """Ranks each name against its own sector, and measures how concentrated the
    top of the overall list is.

    A screen that sorts the whole market on one number will hand you whichever
    sector the macro currently favours. In the live run eight of the top fifty
    were Energy against 2.1 expected, four of them refiners sitting in the top
    fourteen - one bet on crack spreads wearing four tickers, not four
    independent institutional decisions. Sector percentile answers the different
    and more useful question: is this name being accumulated relative to the
    other places that money could sit inside its own sector?
    """
    by_sector: dict[str, list[Metrics]] = {}
    for m in results:
        by_sector.setdefault(m.sector or "Unknown", []).append(m)

    stats = {}
    for sector, group in by_sector.items():
        scores = sorted(m.score for m in group)
        median = float(np.median(scores)) if scores else float("nan")
        stats[sector] = {"n": len(group), "median": round(median, 1),
                         "top": round(max(scores), 1) if scores else None}
        if len(group) < SECTOR_MIN_N:
            continue
        ranked = sorted(group, key=lambda x: x.score)
        for i, m in enumerate(ranked):
            m.sector_pctile = round(100 * i / (len(ranked) - 1), 1)
            m.sector_delta = round(m.score - median, 1)

    # The same, one level down. Sector is too coarse to catch the real problem:
    # four oil refiners are one bet on crack spreads, but GICS calls them all
    # "Energy" alongside pipelines and drillers.
    by_sub: dict[str, list[Metrics]] = {}
    for m in results:
        by_sub.setdefault(m.sub_industry or "Unknown", []).append(m)
    for sub, group in by_sub.items():
        if len(group) < SUB_MIN_N or sub == "Unknown":
            continue
        ranked = sorted(group, key=lambda x: x.score)
        for i, m in enumerate(ranked):
            m.sub_pctile = round(100 * i / (len(ranked) - 1), 1)

    # Concentration of the overall top N against what the field share predicts.
    top = sorted(results, key=lambda x: x.score, reverse=True)[:top_n]
    n = len(results)
    conc = []
    for sector, group in by_sector.items():
        got = sum(1 for m in top if (m.sector or "Unknown") == sector)
        expected = top_n * len(group) / n
        if got and expected > 0 and got >= 1.6 * expected and got >= 3:
            conc.append({"sector": sector, "got": got,
                         "expected": round(expected, 1),
                         "ratio": round(got / expected, 1),
                         "tickers": [m.ticker for m in top
                                     if (m.sector or "Unknown") == sector]})
    conc.sort(key=lambda d: d["ratio"], reverse=True)

    # Sub-industry clusters in the top N. Three names from one sub-industry is
    # already a cluster worth naming, whatever the expected share says.
    sub_conc = []
    for sub, group in by_sub.items():
        if sub == "Unknown":
            continue
        got = [m.ticker for m in top if (m.sub_industry or "Unknown") == sub]
        if len(got) >= 3:
            sub_conc.append({"sub_industry": sub, "got": len(got),
                             "expected": round(top_n * len(group) / n, 1),
                             "tickers": got})
    sub_conc.sort(key=lambda d: d["got"], reverse=True)
    return {"sectors": stats, "concentration": conc,
            "sub_concentration": sub_conc, "top_n": top_n}


def analyze(ticker: str, df: pd.DataFrame, name: str, sector: str,
            sub_industry: str = "Unknown") -> Metrics | None:
    if len(df) < 220:
        return None
    close, vol = df["Close"], df["Volume"]

    m = Metrics(ticker=ticker, name=name, sector=sector,
                sub_industry=sub_industry)
    m.price = float(close.iloc[-1])
    m.avg_dollar_vol = float((close * vol).tail(50).mean())

    m.ud_ratio = up_down_volume_ratio(df)
    (m.acc_weeks, m.dist_weeks, m.acc_streak,
     m.week_pattern, m.week_pressure) = weekly_pattern(df)
    m.ad_slope = ad_line_slope(df)
    m.pullback_dryness = pullback_dryness(df)
    v50, v200 = float(vol.tail(50).mean()), float(vol.tail(200).mean())
    m.vol_expansion = v50 / v200 if v200 > 0 else float("nan")
    m.dist_days = distribution_days(df)

    ma50, ma200 = float(close.tail(50).mean()), float(close.tail(200).mean())
    hi52 = float(close.tail(252).max())
    m.pct_vs_50dma = m.price / ma50 - 1 if ma50 else float("nan")
    m.pct_vs_200dma = m.price / ma200 - 1 if ma200 else float("nan")
    m.pct_off_high = m.price / hi52 - 1 if hi52 else float("nan")
    m.ret_63d = float(close.iloc[-1] / close.iloc[-64] - 1) if len(close) > 64 else float("nan")

    m = score_stock(m)
    m.art_score = article_score(m)
    return m


# ----------------------------------------------------------------------------
# Market health - the article applies distribution days to the indexes
# ----------------------------------------------------------------------------

def market_health() -> list[dict]:
    import yfinance as yf
    out = []
    for sym, label in [("^GSPC", "S&P 500"), ("^IXIC", "Nasdaq Composite")]:
        try:
            df = yf.download(sym, period="6mo", interval="1d",
                             auto_adjust=True, progress=False)
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            df = df.dropna(subset=["Close", "Volume"])
            n = distribution_days(df, threshold=DIST_DAY_DROP)  # the literal rule
            if n <= 2:
                state, note = "good", "Institutions are not exiting."
            elif n <= 4:
                state, note = "warning", "Watch. Pressure is building."
            else:
                state, note = "critical", "Heavy selling. Be defensive."
            out.append({"label": label, "count": n, "state": state, "note": note,
                        "last": float(df["Close"].iloc[-1]),
                        "chg": float(df["Close"].iloc[-1] / df["Close"].iloc[-2] - 1)})
        except Exception as exc:                                  # noqa: BLE001
            print(f"  market health {sym} failed: {exc}")
    return out


# ----------------------------------------------------------------------------
# The screen itself. main() is a thin argparse wrapper around this, and the
# Colab notebook calls it directly - one code path, one thing to test.
# ----------------------------------------------------------------------------

def run_screen(universe: str = "broad", tickers_file: str | None = None,
               outdir: str = "output", min_price: float = 7.0,
               min_dollar_vol: float = 5e6, period: str = "2y",
               batch: int = 100, limit: int | None = None) -> dict:
    """Runs the full screen and writes the report. Returns the payload dict."""
    os.makedirs(outdir, exist_ok=True)

    print("Building universe ...")
    uni = (load_tickers_file(tickers_file) if tickers_file
           else build_universe(universe))
    if limit:
        uni = uni.head(limit)
    print(f"  {len(uni)} tickers\n")

    print("Downloading price history (this is the slow part) ...")
    prices = download_prices(uni["ticker"].tolist(), period=period, batch=batch)
    print(f"  usable history for {len(prices)} of {len(uni)}\n")

    if not prices:
        raise RuntimeError("No price data came back. Check the network "
                           "connection to Yahoo Finance and try again.")

    print("Scoring ...")
    meta = uni.set_index("ticker")
    results: list[Metrics] = []
    for t, df in prices.items():
        try:
            row = meta.loc[t]
            m = analyze(t, df, str(row["name"]), str(row["sector"]),
                        str(row.get("sub_industry", "Unknown")))
        except Exception:                                         # noqa: BLE001
            continue
        if m is None:
            continue
        if m.price < min_price or m.avg_dollar_vol < min_dollar_vol:
            continue
        results.append(m)

    print(f"  {len(results)} passed the liquidity filter")

    # Cross-sectional passes: banding, the gate, and the momentum residual all
    # need the whole field, so they happen once here rather than per stock.
    classify(results)
    for i, m in enumerate(sorted(results, key=lambda x: x.art_score, reverse=True), 1):
        m.art_rank = i
    fit = add_momentum_residual(results)
    sectors = add_sector_ranks(results)
    gated = sum(1 for m in results if m.gated)
    if fit:
        print(f"  score vs 63-day return: R^2 {fit['r2']:.2f} "
              f"(the ex-momentum column strips this out)")
    if gated:
        print(f"  {gated} names capped by the A/D gate "
              f"(labelled accumulation with a falling A/D line)")
    for cx in sectors.get("sub_concentration", []):
        print(f"  cluster: {cx['got']} of the top {sectors['top_n']} are "
              f"{cx['sub_industry']} - {', '.join(cx['tickers'][:6])}")
    for cx in sectors["concentration"]:
        print(f"  concentration: {cx['got']} of the top {sectors['top_n']} are "
              f"{cx['sector']} vs {cx['expected']} expected ({cx['ratio']}x) - "
              f"{', '.join(cx['tickers'][:6])}"
              f"{' ...' if len(cx['tickers']) > 6 else ''}")
    print()

    results.sort(key=lambda x: x.score, reverse=True)
    for i, m in enumerate(results, 1):
        m.components["_rank"] = i

    print("Checking market health ...")
    health = market_health()
    for h in health:
        print(f"  {h['label']}: {h['count']} distribution days in "
              f"the last {DIST_DAY_WINDOW} sessions - {h['note']}")

    rows = [asdict(m) for m in results]
    for r in rows:
        r["rank"] = r["components"].pop("_rank")

    df_out = pd.DataFrame([{k: v for k, v in r.items()
                            if k not in ("components", "week_pattern")}
                           for r in rows])
    csv_path = os.path.join(outdir, "screen_results.csv")
    df_out.to_csv(csv_path, index=False)

    payload = {
        "generated": datetime.now(timezone.utc).astimezone().strftime("%b %d, %Y %I:%M %p %Z"),
        "universe": tickers_file or universe,
        "count": len(rows),
        "scanned": len(prices),
        "params": {"ud_window": UD_WINDOW, "week_lookback": WEEK_LOOKBACK,
                   "universe_set": UNIVERSE_SETS.get(universe, []),
                   "dist_day_window": DIST_DAY_WINDOW,
                   "dist_day_sigma": DIST_DAY_SIGMA,
                   "min_price": min_price, "min_dollar_vol": min_dollar_vol},
        "fit": fit,
        "gated": gated,
        "sectors": sectors,
        "health": health,
        "rows": rows,
    }
    json_path = os.path.join(outdir, "screen_results.json")
    with open(json_path, "w") as fh:
        json.dump(payload, fh, default=float)

    html_path = os.path.join(outdir, "institutional_screener.html")
    write_report(payload, html_path)

    print(f"\nWrote:\n  {html_path}\n  {csv_path}\n  {json_path}")
    print("\nTop 10 by institutional accumulation score:")
    for m in results[:10]:
        print(f"  {m.ticker:<6} {m.score:5.1f}  U/D {m.ud_ratio:4.2f}  "
              f"acc wks {m.acc_weeks}/{WEEK_LOOKBACK}  {m.classification}")

    ex = sorted((m for m in results if np.isfinite(m.ex_mom) and not m.gated),
                key=lambda x: x.ex_mom, reverse=True)[:10]
    if ex:
        print("\nTop 10 ex-momentum, gate-clean "
              "(accumulation the price has not shown yet):")
        for m in ex:
            print(f"  {m.ticker:<6} ex-mom {m.ex_mom:+5.1f}  score {m.score:5.1f}  "
                  f"63d return {m.ret_63d:+6.1%}")

    art = sorted(results, key=lambda x: x.art_score, reverse=True)[:10]
    print("\nTop 10 on the article's own criteria (independent of the composite):")
    for m in art:
        print(f"  {m.ticker:<6} article {m.art_score:5.1f}  composite {m.score:5.1f}  "
              f"U/D {m.ud_ratio:4.2f}  streak {m.acc_streak}")

    best = {}
    for m in results:
        if np.isfinite(m.sector_pctile):
            cur = best.get(m.sector)
            if cur is None or m.score > cur.score:
                best[m.sector] = m
    if best:
        print("\nBest name in each sector (the de-clustered list):")
        for sector, m in sorted(best.items(), key=lambda kv: -kv[1].score):
            print(f"  {sector:<24} {m.ticker:<6} score {m.score:5.1f}  "
                  f"ex-mom {m.ex_mom:+5.1f}  {m.classification}")
    return payload


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--universe", default="broad",
                    choices=list(UNIVERSE_SETS),
                    help="broad = S&P 500+400+NDX (~1,000); "
                         "wide adds S&P 600 small caps (~1,600)")
    ap.add_argument("--tickers-file", help="one ticker per line; overrides --universe")
    ap.add_argument("--outdir", default="output")
    ap.add_argument("--min-price", type=float, default=7.0)
    ap.add_argument("--min-dollar-vol", type=float, default=5e6,
                    help="minimum 50-day average dollar volume")
    ap.add_argument("--period", default="2y")
    ap.add_argument("--batch", type=int, default=100)
    ap.add_argument("--limit", type=int, help="cap universe size (for a quick test)")
    args = ap.parse_args()
    try:
        run_screen(universe=args.universe, tickers_file=args.tickers_file,
                   outdir=args.outdir, min_price=args.min_price,
                   min_dollar_vol=args.min_dollar_vol, period=args.period,
                   batch=args.batch, limit=args.limit)
    except RuntimeError as exc:
        print(exc)
        return 1
    return 0


In [ ]:
#@title Step 4 — settings  *(the defaults are fine — skip this)* { display-mode: "form" }

#@markdown **Universe.** `broad` = S&P 500 + 400 + Nasdaq-100 (~1,000 names, 5-10 min).
#@markdown `wide` adds S&P 600 small caps (~1,600 names, 10-20 min) — where
#@markdown institutional footprints are easiest to see, because one fund buying
#@markdown actually moves the volume.
universe = "broad" #@param ["broad", "wide", "sp500", "sp400", "sp600", "ndx"]

#@markdown **Minimum share price** — drops anything cheaper.
min_price = 7 #@param {type:"number"}

#@markdown **Minimum average daily dollar volume, in millions** — drops thin names.
min_dollar_volume_millions = 5 #@param {type:"number"}

#@markdown **Cap the number of names** (0 = no cap). Set to 50 for a quick test run.
limit = 0 #@param {type:"integer"}

SETTINGS = dict(universe=universe,
                min_price=float(min_price),
                min_dollar_vol=float(min_dollar_volume_millions) * 1e6,
                limit=int(limit) or None,
                outdir="output")
print("Settings:", SETTINGS)


In [ ]:
#@title Step 5 — run the screen, preview it, and download it { display-mode: "form" }

import os, html as _html
import pandas as pd
from IPython.display import display, HTML

payload = run_screen(**SETTINGS)

path = "output/institutional_screener.html"

# --- the top of the ranking, inline ---
df = pd.read_csv("output/screen_results.csv")
cols = ["rank", "ticker", "name", "score", "art_score", "ex_mom", "classification",
        "ud_ratio", "acc_streak", "dist_days", "price"]
display(HTML("<h3>Top 20 by institutional accumulation score</h3>"))
display(df[[c for c in cols if c in df.columns]].head(20)
          .style.hide(axis="index")
          .format({"score": "{:.0f}", "art_score": "{:.0f}", "ex_mom": "{:+.1f}",
                   "ud_ratio": "{:.2f}", "price": "${:.2f}"}))

# --- the full interactive report, previewed in place ---
doc = open(path, encoding="utf-8").read()
display(HTML("<h3>Full report</h3>"))
display(HTML(f'<iframe srcdoc="{_html.escape(doc, quote=True)}" '
             f'style="width:100%;height:820px;border:1px solid #ddd;'
             f'border-radius:10px;background:#fff"></iframe>'))

# --- ...and downloaded to your computer ---
try:
    from google.colab import files
    files.download(path)
    files.download("output/screen_results.csv")
    print("\nDownloading institutional_screener.html and screen_results.csv.")
    print("If the browser blocked it: folder icon in the left sidebar -> "
          "output -> right-click the file -> Download.")
except ImportError:
    print(f"\nNot in Colab. Report is at {os.path.abspath(path)}")


---

### What the score means

Six components, 100 points total:

| Component | Points | The idea |
| --- | --- | --- |
| Up/down volume ratio, 50 days | 30 | Up-day volume ÷ down-day volume. Above 2.0 is strong accumulation, 1.0 is churn, below 1.0 means heavier selling. |
| Weekly range + volume | 25 | Weeks closing in the top 35% of their range on heavy volume, with a bonus when they repeat consecutively — the repetition is the signal, not any one week. |
| A/D line slope, 25 days | 20 | The classic Accumulation/Distribution line, normalized by volume so different-sized stocks compare. |
| Pullback volume drying | 10 | Pulling back on *lighter* volume means institutions paused buying — not that they sold. |
| Volume expansion | 5 | 50-day vs 200-day average volume. No footprints, no institutions. |
| Trend / support | 10 | Price above its 50- and 200-day averages and near its 52-week high. |

**Distribution days** — sessions closing 0.2% or more lower on higher volume — are
counted for the S&P 500 and Nasdaq and shown at the top of the report. Read that
gauge first: a high-scoring stock in a market under distribution is a different
proposition than the same stock in a calm one.

In the report, **click any row** to see which components are carrying its score.
Two names at 72 can get there very differently.

### If something goes wrong

- **"No price data came back"** — Yahoo rate-limited the session. Wait a minute and
  re-run step 5, or set `limit` to a smaller number in step 4.
- **The download didn't happen** — click the folder icon in the left sidebar, open
  `output`, right-click `institutional_screener.html` → Download.
- **A name you expected is missing** — it was filtered out by `min_price` or
  `min_dollar_volume_millions`, or it has under a year of trading history.

*Educational tool, not investment advice. These signals describe what price and
volume have already done; they do not predict what a stock will do next.*
